In [1]:
import requests
import pandas as pd
import time

hub_genes = pd.read_csv(r"C:\tri_disease_project\results\tables\hub_genes_final.csv")['gene_symbol'].tolist()
print("Querying DGIdb for", len(hub_genes), "hub genes:", hub_genes)

DGIDB_URL = "https://dgidb.org/api/graphql"

query = """
query($names: [String!]) {
  genes(names: $names) {
    nodes {
      name
      interactions {
        drug {
          name
          approved
        }
        interactionScore
        interactionTypes {
          type
          directionality
        }
        sources {
          sourceDbName
        }
      }
    }
  }
}
"""

response = requests.post(DGIDB_URL, json={"query": query, "variables": {"names": hub_genes}})
print("Status code:", response.status_code)

data = response.json()
print(data.keys())

Querying DGIdb for 17 hub genes: ['PPARG', 'TNF', 'ESR1', 'IL6', 'TP53', 'APOE', 'PTEN', 'BCL2', 'IGF1', 'LEP', 'LPL', 'CCND1', 'CD4', 'PIK3R1', 'SMAD3', 'PPARGC1A', 'BDNF']
Status code: 200
dict_keys(['data'])


In [2]:
records = []
for gene_node in data['data']['genes']['nodes']:
    gene_name = gene_node['name']
    for interaction in gene_node['interactions']:
        records.append({
            'gene': gene_name,
            'drug': interaction['drug']['name'],
            'fda_approved': interaction['drug']['approved'],
            'interaction_score': interaction['interactionScore'],
            'interaction_types': ', '.join([t['type'] for t in interaction['interactionTypes']]) if interaction['interactionTypes'] else '',
            'sources': ', '.join([s['sourceDbName'] for s in interaction['sources']])
        })

drug_gene_df = pd.DataFrame(records)
print("Total drug-gene interactions found:", len(drug_gene_df))
print("Genes with at least one interaction:", drug_gene_df['gene'].nunique(), "/", len(hub_genes))

# summary per gene
summary = drug_gene_df.groupby('gene').agg(
    n_interactions=('drug', 'count'),
    n_approved_drugs=('fda_approved', 'sum')
).sort_values('n_interactions', ascending=False)
print("\nDruggability summary per hub gene:")
print(summary)

# top scoring interactions overall (highest confidence)
print("\nTop 20 highest-confidence drug-gene interactions:")
print(drug_gene_df.sort_values('interaction_score', ascending=False).head(20)[['gene', 'drug', 'fda_approved', 'interaction_score', 'sources']].to_string(index=False))

drug_gene_df.to_csv(r"C:\tri_disease_project\results\tables\dgidb_hub_gene_interactions.csv", index=False)

Total drug-gene interactions found: 1816
Genes with at least one interaction: 16 / 17

Druggability summary per hub gene:
        n_interactions  n_approved_drugs
gene                                    
TP53               460               155
ESR1               248               110
SMAD3              235                28
PPARG              175                39
PTEN               154                66
BDNF               106                71
TNF                104                57
BCL2                83                33
IL6                 61                36
PIK3R1              40                 6
APOE                39                24
LPL                 32                24
CCND1               31                18
LEP                 24                20
CD4                 18                 3
IGF1                 6                 2

Top 20 highest-confidence drug-gene interactions:
gene         drug  fda_approved  interaction_score                                    sou

In [3]:
approved_only = drug_gene_df[drug_gene_df['fda_approved'] == True].sort_values(['gene', 'interaction_score'], ascending=[True, False])

print("FDA-approved drug interactions per hub gene (top 3 per gene by confidence):")
top3_per_gene = approved_only.groupby('gene').head(3)
print(top3_per_gene[['gene', 'drug', 'interaction_score', 'sources']].to_string(index=False))

top3_per_gene.to_csv(r"C:\tri_disease_project\results\tables\dgidb_top_approved_drugs.csv", index=False)

FDA-approved drug interactions per hub gene (top 3 per gene by confidence):
  gene                       drug  interaction_score                                                                   sources
  APOE                     LUTEIN           1.357230                                                                       NCI
  APOE                  LECANEMAB           1.357230                                                                  PharmGKB
  APOE                  DONANEMAB           1.357230                                                                  PharmGKB
  BCL2   EPIRUBICIN HYDROCHLORIDE           0.255094                                                                       NCI
  BCL2                 VENETOCLAX           0.250539                              ChEMBL, CGI, CIViC, TTD, GuideToPharmacology
  BCL2 MITOXANTRONE HYDROCHLORIDE           0.182210                                                                       NCI
  BDNF                VERTEPORFIN  